# 03 — Analyze Results

View SQLite tables, compare reference vs ours, export CSV and TRACe bar chart to Drive.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/capstone-rag-kag"
PRESET = "covidqa_tuned"
DOMAIN = "biomedical"
DATASET = None
SESSION_ID = None  # None = latest session

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

REPO_ROOT = Path(DRIVE_ROOT) / "repo"
sys.path.insert(0, str(REPO_ROOT))

from colab.lib.colab_setup import bootstrap
from colab.lib.analyze import (
    analyze_run,
    compare_dataframe,
    session_summary,
    table_counts,
    plot_trace_metrics,
)
from IPython.display import Image, display

exp = bootstrap(
    drive_root=DRIVE_ROOT,
    preset=PRESET,
    domain=DOMAIN,
    dataset=DATASET,
    install_deps=True,
    mount=False,
)

In [ ]:
print("Table counts:")
for table, n in table_counts(exp.sqlite_db).items():
    print(f"  {table}: {n}")

print("\nLatest session:")
print(session_summary(exp.sqlite_db, SESSION_ID))

In [ ]:
df = compare_dataframe(exp.sqlite_db, SESSION_ID)
cols = [
    "id", "query",
    "ref_relevance", "our_relevance",
    "ref_utilization", "our_utilization",
    "ref_completeness", "our_completeness",
    "ref_adherence", "our_adherence",
    "parse_error",
]
df[cols].head(10)

In [ ]:
result = analyze_run(exp, SESSION_ID)
print(f"CSV:   {result['csv_path']}")
print(f"Chart: {result['chart_path']}")
display(Image(filename=result["chart_path"]))

In [ ]:
# Drill-down: inspect one sample
SAMPLE_ID = 1
row = df[df["id"] == SAMPLE_ID].iloc[0]
print("Question:", row["query"])
print("\nAnswer:", row["response"])
print("\nContext:", row["context"])